# AssemblyGuard — Treinamento TinyML (IESTI01)

Classificação das etapas de montagem do **Bloq Volt** a partir de vídeo,
com deploy final no **XIAO ESP32S3 Sense**.

**Antes de rodar:** faça upload de `assemblyguard_dataset_limpo.zip` no Colab
(ícone de pasta na barra lateral) ou coloque no seu Google Drive.

O dataset já vem:
- com os overlays da câmera Hikvision removidos (sem vazamento de rótulo);
- recortado na área útil e redimensionado para 96×96;
- **dividido por segmento temporal**, não por frame aleatório.

> **Runtime → Alterar tipo de ambiente → GPU** deixa o treino bem mais rápido.


## 1. Setup

In [ ]:
import os, glob, json, zipfile, shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'nenhuma (vai rodar em CPU)')

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Descompacta o dataset enviado
ZIP = 'assemblyguard_dataset_limpo.zip'
assert os.path.exists(ZIP), f'Faça upload de {ZIP} primeiro (ícone de pasta à esquerda)'

if os.path.exists('dataset'):
    shutil.rmtree('dataset')
with zipfile.ZipFile(ZIP) as z:
    z.extractall('.')

for split in ['train', 'test']:
    print(f'\n{split}:')
    for cls in sorted(os.listdir(f'dataset/{split}')):
        n = len(glob.glob(f'dataset/{split}/{cls}/*.jpg'))
        print(f'  {cls:<16} {n:>5}')

## 2. Carregar os dados

O split treino/teste **já está feito nas pastas** e foi construído por segmento
temporal. Não refaça o split aqui — embaralhar os frames destruiria essa
separação e a acurácia ficaria artificialmente alta.

In [ ]:
IMG_SIZE = 96
BATCH = 32

train_full = tf.keras.utils.image_dataset_from_directory(
    'dataset/train', image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, label_mode='categorical', shuffle=True, seed=SEED)

test_ds = tf.keras.utils.image_dataset_from_directory(
    'dataset/test', image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, label_mode='categorical', shuffle=False)

CLASSES = train_full.class_names
N_CLASSES = len(CLASSES)
print('Classes:', CLASSES)

# validação: separa ~15% dos LOTES de treino (o teste continua intocado)
n_val = max(1, int(0.15 * tf.data.experimental.cardinality(train_full).numpy()))
val_ds = train_full.take(n_val)
train_ds = train_full.skip(n_val)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# Pesos por classe — o dataset é desbalanceado (solda tem 8x mais que fim)
counts = np.array([len(glob.glob(f'dataset/train/{c}/*.jpg')) for c in CLASSES], float)
class_weight = {i: counts.sum() / (len(counts) * n) for i, n in enumerate(counts)}
for i, c in enumerate(CLASSES):
    print(f'{c:<16} n={int(counts[i]):>4}  peso={class_weight[i]:.2f}')

## 3. Augmentation

Só transformações que fazem sentido para uma câmera **fixa** sobre a bancada.
Nada de flip horizontal: espelhar a cena inverteria a posição das zonas
(estação de solda, área de empilhagem) e ensinaria o modelo errado.

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomBrightness(0.20),   # variação de iluminação do galpão
    tf.keras.layers.RandomContrast(0.20),
    tf.keras.layers.RandomTranslation(0.05, 0.05),  # leve tremida da câmera
    tf.keras.layers.RandomZoom(0.08),
], name='augment')

# visualiza o efeito
batch = next(iter(train_ds))
plt.figure(figsize=(12, 3))
for i in range(6):
    plt.subplot(1, 6, i+1)
    plt.imshow(tf.cast(augment(batch[0][i:i+1], training=True)[0], tf.uint8))
    plt.title(CLASSES[np.argmax(batch[1][i])], fontsize=8)
    plt.axis('off')
plt.tight_layout(); plt.show()

## 4. Modelo

MobileNetV2 com `alpha=0.35` — a mesma família que o Edge Impulse usa nos
blocos de transfer learning, dimensionada para caber no ESP32S3.

In [ ]:
def build_model():
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        alpha=0.35, include_top=False, weights='imagenet')
    base.trainable = False

    inp = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augment(inp)
    x = tf.keras.layers.Rescaling(1./127.5, offset=-1)(x)   # MobileNetV2 espera [-1,1]
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(N_CLASSES, activation='softmax')(x)

    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
    return m, base

model, base = build_model()
model.summary()

In [ ]:
cbs = [
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True,
                                     monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=4, min_lr=1e-6),
]

hist = model.fit(train_ds, validation_data=val_ds, epochs=40,
                 class_weight=class_weight, callbacks=cbs, verbose=1)

### Fine-tuning

Descongela as últimas camadas com learning rate baixo. Ganha alguns pontos,
mas cuidado: com dataset pequeno é fácil sobreajustar.

In [ ]:
base.trainable = True
for l in base.layers[:-25]:
    l.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])

hist_ft = model.fit(train_ds, validation_data=val_ds, epochs=20,
                    class_weight=class_weight, callbacks=cbs, verbose=1)

In [ ]:
acc = hist.history['accuracy'] + hist_ft.history['accuracy']
vacc = hist.history['val_accuracy'] + hist_ft.history['val_accuracy']
plt.figure(figsize=(6,4))
plt.plot(acc, label='treino'); plt.plot(vacc, label='validação')
plt.axvline(len(hist.history['accuracy'])-0.5, ls='--', c='gray', lw=1)
plt.xlabel('época'); plt.ylabel('acurácia'); plt.legend(); plt.grid(alpha=.3)
plt.title('Curva de treino (linha = início do fine-tuning)'); plt.show()

## 5. Avaliação no conjunto de teste

Este é o número que vale — os frames de teste vêm de **segmentos temporais
que o modelo nunca viu**.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for xb, yb in test_ds:
    p = model.predict(xb, verbose=0)
    y_true.extend(np.argmax(yb, 1)); y_pred.extend(np.argmax(p, 1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7,6))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(N_CLASSES), CLASSES, rotation=45, ha='right')
plt.yticks(range(N_CLASSES), CLASSES)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        plt.text(j, i, cm[i,j], ha='center',
                 color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.xlabel('predito'); plt.ylabel('real'); plt.title('Matriz de confusão (teste)')
plt.colorbar(); plt.tight_layout(); plt.show()

## 6. Teste de vazamento (control experiment)

**Não pule esta etapa** — é o que prova que o modelo aprendeu o processo e
não algum artefato da imagem.

A ideia: treinar um modelo idêntico com os **rótulos embaralhados**. Se ele
também acertar bem, existe algum atalho no dataset (vazamento). Se ficar perto
do acaso (~1/7 ≈ 14%), o pipeline está limpo.

In [ ]:
def dataset_rotulos_embaralhados(ds):
    labels = np.concatenate([y.numpy() for _, y in ds])
    rng = np.random.default_rng(SEED); rng.shuffle(labels)
    imgs = np.concatenate([x.numpy() for x, _ in ds])
    return tf.data.Dataset.from_tensor_slices((imgs, labels)).batch(BATCH).prefetch(AUTOTUNE)

ctrl_train = dataset_rotulos_embaralhados(train_ds)
ctrl, _ = build_model()
ctrl.fit(ctrl_train, epochs=12, verbose=0)

ctrl_acc = ctrl.evaluate(test_ds, verbose=0)[1]
real_acc = model.evaluate(test_ds, verbose=0)[1]
acaso = 1.0 / N_CLASSES

print(f'acaso            : {acaso:.1%}')
print(f'controle (shuffle): {ctrl_acc:.1%}')
print(f'modelo real       : {real_acc:.1%}')
print()
if ctrl_acc > acaso * 1.8:
    print('ALERTA: o controle acertou bem acima do acaso -> ainda há atalho no dataset.')
else:
    print('OK: o controle ficou perto do acaso -> sem vazamento aparente.')

## 7. Quantização int8 e exportação

O ESP32S3 roda int8. A quantização precisa de um *representative dataset*
com imagens reais para calibrar as escalas.

In [ ]:
def rep_data():
    for xb, _ in train_ds.take(20):
        for i in range(xb.shape[0]):
            yield [tf.expand_dims(xb[i], 0)]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8

tflite_int8 = conv.convert()
open('assemblyguard_int8.tflite', 'wb').write(tflite_int8)
print(f'modelo int8: {len(tflite_int8)/1024:.1f} KB')

In [ ]:
# Confere se a quantização degradou a acurácia
interp = tf.lite.Interpreter(model_content=tflite_int8)
interp.allocate_tensors()
inp_d = interp.get_input_details()[0]; out_d = interp.get_output_details()[0]
scale, zp = inp_d['quantization']

acertos = total = 0
for xb, yb in test_ds:
    for i in range(xb.shape[0]):
        q = np.clip(xb[i].numpy()/scale + zp, -128, 127).astype(np.int8)
        interp.set_tensor(inp_d['index'], q[None, ...])
        interp.invoke()
        pred = np.argmax(interp.get_tensor(out_d['index'])[0])
        acertos += int(pred == np.argmax(yb[i].numpy())); total += 1

print(f'acurácia float32: {real_acc:.1%}')
print(f'acurácia int8   : {acertos/total:.1%}')
print(f'queda           : {(real_acc - acertos/total)*100:.1f} pontos')

In [ ]:
# Gera o array C para embutir no firmware do ESP32
os.system('xxd -i assemblyguard_int8.tflite > model_data.cc')
with open('model_data.cc') as f:
    head = f.read(400)
print(head, '...')

json.dump({'classes': CLASSES, 'img_size': IMG_SIZE,
           'acuracia_teste_float': float(real_acc),
           'acuracia_teste_int8': acertos/total},
          open('model_info.json','w'), indent=2, ensure_ascii=False)

from google.colab import files
files.download('assemblyguard_int8.tflite')
files.download('model_data.cc')
files.download('model_info.json')

## 8. Próximos passos

1. **Critérios de sucesso** — comparem os números da seção 5 com as metas do
   projeto (acurácia mínima por classe, taxa de falso positivo). Se `fim` ou
   `empilhagem 2` ficarem fracas, é falta de exemplo: gravem mais sessões.

2. **Validação em vídeo novo** — rodem o modelo num vídeo de outra sessão,
   não só nos frames de teste. É o teste mais próximo do mundo real.

3. **Domínio da câmera** — este dataset veio de uma câmera IP Hikvision; o
   deploy será numa OV2640 do XIAO, com cor, ângulo e nitidez diferentes.
   Contem com uma queda de acurácia e planejem **regravar parte do dataset
   com a própria câmera do ESP32** antes da validação final.

4. **Deploy** — o `.tflite` int8 vai para o firmware (`AssemblyGuard_firmware.ino`).
   Alternativa: subir este mesmo dataset no Edge Impulse e usar a exportação
   "Arduino library", que já gera o wrapper de inferência pronto.

5. **Sequenciamento** — o classificador diz a etapa atual; a lógica de
   comparação com a ordem esperada (Ewerton) é que gera o alerta de desvio.
